# Turntable: minimal interface demo

This notebook walks through the `Residuals` / `Segment` / `Wheel` contract using `EchoSegment`, a no-op segment that just prints what the Wheel hands it. No real waveform model, no MCMC — just enough to see the plumbing work.

## 1. Build the observed `Residuals`

One frozen object holds both the TDI arrays and the run settings everyone in this run agrees on.

In [ ]:
import numpy as np

from turntable import Residuals, Wheel
from turntable.testing import EchoSegment

rng = np.random.default_rng(0)
n_samples = 1024
channels = ("A", "E", "T")

observed = Residuals(
    tdi={ch: rng.standard_normal(n_samples) for ch in channels},
    sample_rate=0.1,
    n_samples=n_samples,
    channels=channels,
    tdi_generation="2.0",
    observable="fractional_frequency",
    epoch=0.0,
)
observed

## 2. Long names and short shadows

Every derived quantity has two spellings. Use whichever reads better.

In [ ]:
print(f"observation_time = {observed.observation_time} s     (Tobs = {observed.Tobs})")
print(f"sample_rate      = {observed.sample_rate} Hz   (fs   = {observed.fs})")
print(f"sample_interval  = {observed.sample_interval} s     (dt   = {observed.dt})")
print(f"n_samples        = {observed.n_samples}        (N    = {observed.N})")
print(f"freq_resolution  = {observed.frequency_resolution} Hz   (df   = {observed.df})")
print(f"nyquist          = {observed.nyquist_frequency} Hz   (fny  = {observed.fny})")
print(f"epoch            = {observed.epoch} s     (t0   = {observed.t0})")

In [ ]:
Residuals.aliases()

## 3. Typo catcher

Common misspellings point at the canonical spelling instead of failing silently.

In [ ]:
try:
    observed.T_obs
except AttributeError as e:
    print(e)

## 4. Plug two `EchoSegment`s into a `Wheel`

Each segment will print what it sees on every Gibbs iteration. Because echo segments render zeros, the residuals each segment sees are just the observed data.

In [ ]:
wheel = Wheel(observed)
wheel.add(EchoSegment(name="ucb"))
wheel.add(EchoSegment(name="mbhb"))

wheel.run(n_iterations=3)

## 5. Inspect per-segment state

Catalogs and states are stored on the Wheel and retrievable by name.

In [ ]:
print("ucb  catalog:", wheel.catalog("ucb"),  " state:", wheel.state("ucb"))
print("mbhb catalog:", wheel.catalog("mbhb"), " state:", wheel.state("mbhb"))